# Session 2 — Databricks Data Quality Monitoring Setup

**Goal:** Configure Data Profiling for deep metrics and drift monitoring on an inference log table.

Data Profiling will:
- Compare the current inference distribution vs a baseline (training data)
- Compute drift metrics (PSI) for every feature
- Track model performance metrics over time
- Auto-generate a monitoring dashboard in Unity Catalog

In [0]:
%run ../utils/config

## Databricks Data Quality Monitoring
Databricks Data Quality Monitoring (DQM) is a managed, serverless capability in Unity Catalog that automatically monitors the freshness, completeness, and statistical quality of your tables at scale, without modifying existing pipelines. It consists of:

- **Anomaly detection** – one‑click, schema‑level monitoring.
- **Data profiling** – deep metrics and drift monitoring for selected tables (formerly Lakehouse Monitoring).

We will focus on Data Profiling today.

## Data Profiling for Inference
Inputs:
- Inference Table - Contains the request log for the model
- Baseline Table - Known "good" data for comparison

Produces:
- Profile Metrics Table - Profile statistics
- Drift Table - Changes in profile metrics over time
- Dashboard - Visualize statistics and drift
- Alerts - Actively detect changes

<img src=../utils/resources/DataProfiling.png>

In [0]:
# Use the simulated inference log (with drifted data) from the previous module
inference_table = f"{catalog}.{schema}.churn_inference_log" # Must be in three level namespace format
print(f"Monitoring table : {inference_table}")
print(f"Baseline table   : {catalog}.`00_shared`.telco_churn")

## Check the Inference Log

Before creating the monitor, verify the inference log has the expected shape.

In [0]:
df_log = spark.table(inference_table)
print(f"Rows: {df_log.count():,}")
print(f"Columns: {df_log.columns}")
display(df_log.limit(3))

## Create the Data Profiling Monitor

This creates a monitor with:
- **InferenceLog** profile type: tracks prediction distribution + model accuracy
- **Baseline**: a view of <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">vectorcatalog02.00_shared.telco_churn</span> with schema-aligned types
- **Granularity**: daily windows
- **Output**: auto-creates <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_inference_log_profile_metrics</span> and <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_inference_log_drift_metrics</span> tables

**Note:** The Data Profiling Monitor requires the baseline table schema to be compatible with the inference log. The raw `telco_churn` table stores `TotalCharges` as STRING (11 rows have empty values in the IBM dataset). We create a view that casts it to DOUBLE to match the inference log schema.

In [0]:
import mlflow
from mlflow import MlflowClient

# Use Unity Catalog registry
mlflow.set_registry_uri("databricks-uc")
uc_client = MlflowClient()

model_name = f"{catalog}.{schema}.churn_classifier"
baseline_view = f"{catalog}.{schema}.churn_monitor_baseline"

# Derive model_id dynamically from the @champion alias
try:
    mv = uc_client.get_model_version_by_alias(model_name, "champion")
    model_id = f"{model_name}_v{mv.version}"
except Exception:
    model_id = f"{model_name}_v1"  # fallback if alias not yet set

print(f"Using model_id: {model_id}")

# Lakehouse Monitoring requires the baseline to have compatible column types.
# telco_churn.TotalCharges is STRING (IBM CSV has 11 empty values);
# churn_inference_log.TotalCharges is DOUBLE. We create a type-aligned baseline view.
spark.sql(f"""
    CREATE OR REPLACE VIEW {baseline_view} AS
    SELECT
        customerID,
        gender,
        SeniorCitizen,
        Partner,
        Dependents,
        TRY_CAST(tenure AS BIGINT),
        PhoneService,
        MultipleLines,
        InternetService,
        OnlineSecurity,
        OnlineBackup,
        DeviceProtection,
        TechSupport,
        StreamingTV,
        StreamingMovies,
        Contract,
        PaperlessBilling,
        PaymentMethod,
        MonthlyCharges,
        TRY_CAST(TotalCharges AS DOUBLE) AS TotalCharges,
        Churn,
        '{model_id}' AS model_id,
        now()        AS inference_ts,
        Churn        AS prediction
    FROM {catalog}.`00_shared`.telco_churn
    WHERE TotalCharges IS NOT NULL
""")

print(f"✓ Baseline view created : {baseline_view}")
print(f"  Row count             : {spark.table(baseline_view).count()}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import (
    MonitorInferenceLog,
    MonitorInferenceLogProblemType,
)

w = WorkspaceClient()
current_user = spark.sql("SELECT current_user()").first()[0]

assets_dir      = f"/Users/{current_user}/monitoring/{schema}"

print(f"Creating monitor on : {inference_table}")
print(f"Assets directory    : {assets_dir}")

try:
    monitor = w.quality_monitors.create(
        table_name=inference_table,
        assets_dir=assets_dir,
        output_schema_name=f"{catalog}.{schema}",
        inference_log=MonitorInferenceLog(
            problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION,
            model_id_col="model_id",
            prediction_col="prediction",
            label_col="Churn",
            timestamp_col="inference_ts",
            granularities=["1 day"],
        ),
        baseline_table_name=baseline_view,
    )
    print("✓ Monitor created!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Monitor already exists — running a refresh instead.")
        refresh = w.quality_monitors.run_refresh(table_name=inference_table)
    else:
        raise

## Refreshing a Quality Monitor
A refresh is initiated when the monitor is first created.  To programmatically initiate a refresh, you can use the following code:  


```
refresh = w.quality_monitors.run_refresh(table_name=inference_table)
print(f"Refresh ID: {refresh.refresh_id}")
```

## Navigate to the Monitor

**In the Databricks UI:**
1. Open **Catalog** in the left sidebar
2. Navigate: <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">vectorcatalog02 → {your_schema} → `churn_inference_log</span>
3. Click the **Quality** tab
4. Click **View refresh history** to see when the monitor was last updated
5. Click **View dashboard** to see a pre-built dashboard for the monitor

Because enough time has not elapsed to gather profiling data, your dashboard will not contain any useful data yet.  A typical Data Profiling dashboard might look like the following:

<img src=../utils/resources/DataProfilingDashboard.png>

The dashboard will include:
  - Filters to control the time slices
  - Inference and performance statistics
  - Drift statistics for predictions, numerical features, and categorical features
  - Data profiling information


## Query the Drift Metrics
The <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">{tablename}_drift_metrics</span> and <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">{tablename}_profile_metrics</span> tables can be queried programmatically.

Note that the drift_metrics table will not be available until the initial refresh of your monitor finishes, which could take 10 minutes or more. 

In [0]:
# Query the drift metrics table directly
# (Available after the refresh completes)

drift_table = f"`{catalog}`.`{schema}`.churn_inference_log_drift_metrics"


if not spark.catalog.tableExists(drift_table):
    print("⏳ Drift metrics table not yet available.")
    print("   The initial monitor refresh is still running (typically 5-10 min).")
    print("   Re-run this cell once the refresh completes.")
else:
    drift_df = spark.table(drift_table)
    row_count = drift_df.count()
    if row_count == 0:
        print("⏳ Drift metrics table exists but is empty — refresh may still be in progress.")
        print("   Re-run this cell in a few minutes.")
    else:
        print(f"✓ Drift metrics table has {row_count} rows")
        display(
            drift_df
            .orderBy("drift_type", "column_name")
            .limit(20)
        )


## AI Gateway: Endpoint Usage Tracking

AI Gateway logs every request to <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">system.serving.endpoint_usage</span> — a system table that
gives you a complete picture of who is calling your endpoint and at what cost.

This data is available with approximately **1-hour latency** and is useful for:
- Auditing: who called the endpoint and when
- Cost attribution: token consumption per user or team
- Capacity planning: peak request rates and latency trends

Note that your account admin must grant permissions on the `system.serving` schema in order for the following query to execute successfully. You will see an error message if you do not have the necessary permissions.  An example of the system table content is included here.

<img src=../utils/resources/endpoint_usage.png>

In [0]:
# Query AI Gateway usage metrics from system tables
# Note: data appears with ~1 hour latency after requests are made

import logging

# get the workspace ID from the notebook context
workspace_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().workspaceId().get()

# Temporarily suppress noisy Spark Connect gRPC error logs
_sc_logger = logging.getLogger("pyspark.sql.connect.logging")
_prev_level = _sc_logger.level
_sc_logger.setLevel(logging.CRITICAL)

try:
    usage_df = spark.sql(f"""
        SELECT
            DATE(request_time)                 AS date,
            e.endpoint_name                    AS endpoint_name,
            COUNT(*)                           AS total_requests,
            SUM(input_token_count)             AS total_input_tokens,
            SUM(output_token_count)            AS total_output_tokens
        FROM system.serving.endpoint_usage u
        JOIN system.serving.served_entities e
            ON u.served_entity_id = e.served_entity_id
        WHERE e.workspace_id = {workspace_id}          
        AND request_time >= current_date() - INTERVAL 7 DAYS
        GROUP BY 1, 2
        ORDER BY 1 DESC
    """)
    print(f"Usage summary for this workspace over the last 7 days:")
    display(usage_df)
except Exception as e:
    error_msg = str(e).lower()
    if "permission" in error_msg or "access denied" in error_msg or "not authorized" in error_msg:
        print("⚠️ You do not have sufficient permissions to query AI Gateway usage metrics.")
        print("   Please contact your workspace administrator to request access to system.serving tables.")
    else:
        print(f"Note: Usage data not yet available (data has ~1hr latency).")
        print(f"Once requests flow through the endpoint, re-run this cell.")
        print(f"Details: {e}")
finally:
    _sc_logger.setLevel(_prev_level)

➡️ Next: [04_drift_alerts.ipynb]($./04_drift_alerts) — create an automated alert that fires when drift is detected